# Woche 9 – Montag: Expected Shortfall (ES) & Kohärenz von Risikomaßen

## Block 1 (06:00 – 08:45): Expected Shortfall – Definition und Berechnung

**Lernziel:** Sie definieren den Expected Shortfall (ES) als Alternative zum VaR.

Der Expected Shortfall zum Konfidenzniveau $\alpha$ ist der bedingte Erwartungswert der Verluste, die den VaR überschreiten:
$$\text{ES}_\alpha(L) = \mathbb{E}[L \mid L \ge \text{VaR}_\alpha(L)].$$

Im Gegensatz zum VaR ist ES **kohärent** (subadditiv, monoton, translationsinvariant, positiv homogen).

**Aufgabe:** Berechnen Sie VaR (95%) und ES für einen normalverteilten Datensatz sowie für Ihre AML-Transaktionsdaten.

In [ ]:
import numpy as np
import pandas as pd

# Normalverteilte Daten
np.random.seed(42)
losses_norm = np.random.normal(0, 1, 1000)
alpha = 0.95
var_norm = np.percentile(losses_norm, alpha * 100)
es_norm = losses_norm[losses_norm >= var_norm].mean()
print(f'Normalverteilt: VaR(95%) = {var_norm:.4f}, ES = {es_norm:.4f}')

# AML-Daten
df = pd.read_json('cleaned_aml_batch.json')
losses_aml = df['amount'].values
var_aml = np.percentile(losses_aml, alpha * 100)
es_aml = losses_aml[losses_aml >= var_aml].mean()
print(f'AML-Daten: VaR(95%) = {var_aml:.2f}, ES = {es_aml:.2f}')


> **Verbesserungshinweis:** Der ES ist immer größer oder gleich dem VaR. Bei nicht‑normalen Verteilungen (z. B. mit schweren Rändern) kann der ES deutlich höher sein.

---

## Block 2 (09:30 – 11:40): Autoencoder – Training mit Hyperparameter‑Optimierung

**Lernziel:** Sie trainieren einen Autoencoder und optimieren Lernrate und Bottleneck‑Dimension.

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# Daten vorbereiten (nur amount und amount_log)
features = df[['amount']].copy()
features['amount_log'] = np.log1p(features['amount'])
scaler = StandardScaler()
X = torch.tensor(scaler.fit_transform(features), dtype=torch.float32)

class AE(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim, bottleneck_dim)
        self.decoder = nn.Linear(bottleneck_dim, input_dim)
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Hyperparameter-Grid
learning_rates = [0.001, 0.01, 0.1]
bottlenecks = [1, 2, 3]
best_loss = float('inf')
best_params = None

for lr in learning_rates:
    for b in bottlenecks:
        model = AE(input_dim=2, bottleneck_dim=b)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.MSELoss()
        for epoch in range(200):
            out = model(X)
            loss = criterion(out, X)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_params = {'lr': lr, 'bottleneck': b}
            torch.save(model.state_dict(), 'best_ae_model.pth')
print(f'Beste Parameter: {best_params}, Loss: {best_loss:.4f}')


> **Verbesserungshinweis:** Eine zu kleine Bottleneck‑Dimension kann Informationen verlieren, eine zu große kann überanpassen. Die optimale Dimension hängt vom Datensatz ab.

---

## Block 3 (12:40 – 14:10): AI Act – Art. 27 (Grundrechte‑Folgenabschätzung)

**Lernziel:** Sie kennen die Pflicht zur Durchführung einer Grundrechte‑Folgenabschätzung für Hochrisiko‑KI.

**Art. 27** verlangt, dass Betreiber von Hochrisiko‑KI vor der Inbetriebnahme eine **Grundrechte‑Folgenabschätzung** durchführen. Diese muss systematisch die Auswirkungen des KI‑Systems auf Grundrechte (z. B. Datenschutz, Nichtdiskriminierung, Meinungsfreiheit) bewerten.

**Aufgabe (GOV):** Notieren Sie die gesetzlich geforderten Inhalte einer solchen Abschätzung (siehe Art. 27 Abs. 2).

> **Verbesserungshinweis:** Die Folgenabschätzung muss vor dem Einsatz dokumentiert werden und regelmäßig aktualisiert werden.

---

## Block 4 (14:40 – 16:50): Zustandspassiv (Buscha Kap. 1.5.1 Vertiefung)

**Lernziel:** Sie unterscheiden Vorgangspassiv („wird gemacht“) und Zustandspassiv („ist gemacht“).

Das **Zustandspassiv** beschreibt das Ergebnis einer Handlung (Zustand), nicht die Handlung selbst. Es wird mit „sein“ + Partizip II gebildet.

Beispiel: „Die Konformitätsbewertung **wird durchgeführt**“ (Vorgang) vs. „Die Konformitätsbewertung **ist durchgeführt**“ (Zustand).

**Übung:** Wandeln Sie folgende Sätze in das Zustandspassiv um:
1. Das Modell wird trainiert. → Das Modell ist trainiert.
2. Die Dokumentation wird erstellt.
3. Die Risiken werden bewertet.

**Schreibübung:** Verfassen Sie einen kurzen Absatz (5 Sätze) über den Status Ihres AML‑Projekts (z. B. „Der Autoencoder ist implementiert, die technische Dokumentation ist erstellt“). Speichern Sie als `41_zustandspassiv_uebung.md` in Track_C.